In [6]:
import sys
import subprocess
from importlib.metadata import version

requirements = [
    "qiskit==2.2.3",
    "numpy>=1.26,<3",
    "scipy>=1.11,<2",
    "matplotlib>=3.8,<4",
    "packaging>=23",
    "torch>=2.5,<3",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", *requirements], check=True
)
print("Python:", sys.version)
print(
    "패키지:",
    {
        name: version(name)
        for name in ["qiskit", "numpy", "scipy", "matplotlib", "torch"]
    },
)
# 새 Python 프로세스에서 import와 설치된 직접 의존성을 확인합니다.
check = r"""
from importlib.metadata import version, requires
from packaging.requirements import Requirement
import qiskit, numpy, scipy, matplotlib
import torch

packages = ["qiskit", "numpy", "scipy", "matplotlib", "torch"]
for name in packages:
    for text in requires(name) or []:
        dependency = Requirement(text)
        if dependency.marker is None or dependency.marker.evaluate():
            actual = version(dependency.name)
            assert not dependency.specifier or dependency.specifier.contains(
                actual, prereleases=True
            ), (name, str(dependency), actual)
print("패키지 import 및 직접 의존성 확인: 통과")
"""
subprocess.run([sys.executable, "-c", check], check=True)

Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
패키지: {'qiskit': '2.2.3', 'numpy': '2.4.6', 'scipy': '1.16.3', 'matplotlib': '3.10.0', 'torch': '2.14.1+cpu'}


CompletedProcess(args=['/usr/bin/python3', '-c', '\nfrom importlib.metadata import version, requires\nfrom packaging.requirements import Requirement\nimport qiskit, numpy, scipy, matplotlib\nimport torch\n\npackages = ["qiskit", "numpy", "scipy", "matplotlib", "torch"]\nfor name in packages:\n    for text in requires(name) or []:\n        dependency = Requirement(text)\n        if dependency.marker is None or dependency.marker.evaluate():\n            actual = version(dependency.name)\n            assert not dependency.specifier or dependency.specifier.contains(\n                actual, prereleases=True\n            ), (name, str(dependency), actual)\nprint("패키지 import 및 직접 의존성 확인: 통과")\n'], returncode=0)

In [7]:
import numpy as np

origin = np.array([0.0, 0.0])
destination = np.array([3.0, -1.0])
distance = float(np.linalg.norm(destination - origin))
speed = 35.0
weather_delay = 0.1
inner = 4.0
floor = 15
packages = 7
eta = 60 * distance / speed * (1 + weather_delay) + inner + 0.1 * floor + 0.2 * packages
print("거리(km):", round(distance, 6), "교육용 ETA(분):", round(eta, 6))

거리(km): 3.162278 교육용 ETA(분): 12.863152


In [8]:
import numpy as np

eta = np.array(
    [[0, 13, 7, 15], [6, 0, 9, 12], [5, 14, 0, 12], [10, 14, 9, 0]], dtype=float
)
x = np.eye(3, dtype=int)
penalty = 100.0  # lambda(람다): 등식 제약의 벌점 계수
cost = sum(eta[0, i + 1] * x[i, 0] + eta[i + 1, 0] * x[i, 2] for i in range(3))
for t in range(2):
    for i in range(3):
        for j in range(3):
            cost += eta[i + 1, j + 1] * x[i, t] * x[j, t + 1]
row = x.sum(axis=1)
column = x.sum(axis=0)
qubo = cost + penalty * (((row - 1) ** 2).sum() + ((column - 1) ** 2).sum())
print("고객별 방문/순서별 배정:", row.tolist(), column.tolist())
print("경로 비용/QUBO:", cost, qubo)
assert cost == 44 and qubo == 44

고객별 방문/순서별 배정: [1, 1, 1] [1, 1, 1]
경로 비용/QUBO: 44.0 44.0


In [9]:
import numpy as np

# 비용 x0+2*x1, 정확히 하나 선택, lambda(람다)=3
c = 3.0
a = np.array([-2.0, -1.0])
B = np.array([[0.0, 6.0], [0.0, 0.0]])
bits = (np.arange(4)[:, None] >> np.arange(2)) & 1
z = 1 - 2 * bits
h = -a / 2 - (B.sum(axis=0) + B.sum(axis=1)) / 4
J = B / 4
constant = c + a.sum() / 2 + B.sum() / 4
qubo = c + bits @ a + np.einsum("bi,ij,bj->b", bits, B, bits)
ising = constant + z @ h + np.einsum("bi,ij,bj->b", z, J, z)
print("h:", h, "J01:", J[0, 1], "상수:", constant)
print("QUBO:", qubo, "Ising:", ising)
assert np.allclose(qubo, ising)

h: [-0.5 -1. ] J01: 1.5 상수: 3.0
QUBO: [3. 1. 2. 6.] Ising: [3. 1. 2. 6.]


In [10]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

# x0+x1=1, 비용 1*x0+2*x1, lambda(람다)=3
bits = (np.arange(4)[:, None] >> np.arange(2)) & 1
cost = bits @ np.array([1.0, 2.0]) + 3 * (bits.sum(axis=1) - 1) ** 2
h = np.array([-0.5, -1.0])
J = 1.5
constant = 3.0
z = 1 - 2 * bits
assert np.allclose(cost, constant + z @ h + J * z[:, 0] * z[:, 1])
gamma = np.pi / 4  # gamma(감마): 비용 위상 계수
beta = np.pi / 8  # beta(베타): 믹서 회전량
qc = QuantumCircuit(2)
qc.h(range(2))
initial = Statevector.from_instruction(qc)
qc.rz(2 * gamma * h[0], 0)
qc.rz(2 * gamma * h[1], 1)
qc.rzz(2 * gamma * J, 0, 1)
after_cost = Statevector.from_instruction(qc)
# 상수항 위상은 모든 진폭에 같은 위상을 곱합니다.
phase_exact = initial.data * np.exp(-1j * gamma * cost)
assert np.allclose(after_cost.data * np.exp(-1j * gamma * constant), phase_exact)
qc.rx(2 * beta, 0)
qc.rx(2 * beta, 1)
final = Statevector.from_instruction(qc)
print("기저 00,01,10,11 비용:", cost.tolist())
print("초기 확률:", np.round(initial.probabilities(), 6))
print("비용 회로 후 진폭(상수항 포함):", np.round(phase_exact, 6))
print("비용 회로 후 확률:", np.round(after_cost.probabilities(), 6))
print("믹서 후 확률:", np.round(final.probabilities(), 6))
print(
    "초기/최종 기대비용:", initial.probabilities() @ cost, final.probabilities() @ cost
)

기저 00,01,10,11 비용: [3.0, 1.0, 2.0, 6.0]
초기 확률: [0.25 0.25 0.25 0.25]
비용 회로 후 진폭(상수항 포함): [-0.353553-0.353553j  0.353553-0.353553j  0.      -0.5j
 -0.      +0.5j     ]
비용 회로 후 확률: [0.25 0.25 0.25 0.25]
믹서 후 확률: [0.614277 0.135723 0.0625   0.1875  ]
초기/최종 기대비용: 2.9999999999999987 3.228553390593272


In [11]:
import time
from pathlib import Path
from importlib.metadata import version
import numpy as np
import torch
from torch import nn
from scipy.optimize import minimize
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

# 1. 실행 환경과 기본 설정
started = time.perf_counter()
print(
    "버전:",
    {k: version(k) for k in ["qiskit", "numpy", "scipy", "torch", "matplotlib"]},
)
torch.set_num_threads(1)
torch.manual_seed(7)

# 2. 변경 가능한 실험 조건
weather = "rain"
vehicle = "motorcycle"
p = 1
maxiter = 100
shots = 4096
penalty = 100.0
speed = {"bike": 15.0, "motorcycle": 35.0, "compact": 25.0, "walking": 4.0}[vehicle]
wdelay = {"clear": 0.0, "rain": 0.1, "snow": 0.3, "hot": 0.05}[weather]

# 3. 배송지와 구간별 입력 데이터
names = ["H", "C1", "C2", "C3"]
coords = np.array([[0, 0], [3, -1], [1.2, 2.5], [5, 2.0]])
# 허브 복귀 서비스 시간은 0분입니다.
inner = np.array([0, 4, 1, 2.5])
floor = np.array([0, 15, 2, 5])
pkg = np.array([0, 7, 2, 10])
pairs = [(i, j) for i in range(4) for j in range(4) if i != j]
features = np.array(
    [
        [
            np.linalg.norm(coords[i] - coords[j]),
            speed,
            wdelay,
            inner[j],
            floor[j],
            pkg[j],
        ]
        for i, j in pairs
    ],
    dtype=np.float32,
)
# 교육용 산식: 거리/속도*60*(1+기상 지연)+건물+층수+물량
labels = (
    features[:, 0] / features[:, 1] * 60 * (1 + features[:, 2])
    + features[:, 3]
    + 0.1 * features[:, 4]
    + 0.2 * features[:, 5]
)

# 4. 특징 정규화와 ETA 신경망 학습
span = np.ptp(features, axis=0)
X = torch.tensor((features - features.min(axis=0)) / np.where(span == 0, 1, span))
y = torch.tensor(labels[:, None] / 20.0)
model = nn.Sequential(
    nn.Linear(6, 16),
    nn.Tanh(),
    nn.Linear(16, 8),
    nn.Tanh(),
    nn.Linear(8, 1),
    nn.Softplus(),
)
opt = torch.optim.Adam(model.parameters(), lr=0.01)
losses = []
for epoch in range(600):
    opt.zero_grad()
    loss = nn.functional.mse_loss(model(X), y)
    loss.backward()
    opt.step()
    losses.append(float(loss.detach()))
with torch.no_grad():
    pred = model(X).numpy().ravel() * 20

# 5. 예측 ETA와 교육용 기준 ETA 행렬
eta = np.zeros((4, 4))
reference = np.zeros((4, 4))
for k, (i, j) in enumerate(pairs):
    eta[i, j] = pred[k]
    reference[i, j] = labels[k]
print("학습 MSE(분²):", float(np.mean((pred - labels) ** 2)))
print("ETA 행렬(분), 행=출발 열=도착 H,C1,C2,C3:\n", np.round(eta, 4))
# k=3*i+t: 고객 i(0=C1), 방문순서 t(0=첫번째), 총 9큐비트

# 6. QUBO 계수 구성
n = 9
a = np.zeros(n)
B = np.zeros((n, n))
c = 6 * penalty


def idx(i, t):
    return 3 * i + t


def add(k, l, v):
    if k == l:
        a[k] += v
    else:
        B[min(k, l), max(k, l)] += v


for i in range(3):
    a[idx(i, 0)] += eta[0, i + 1]
    a[idx(i, 2)] += eta[i + 1, 0]
for t in range(2):
    for i in range(3):
        for j in range(3):
            add(idx(i, t), idx(j, t + 1), eta[i + 1, j + 1])
groups = [[idx(i, t) for i in range(3)] for t in range(3)] + [
    [idx(i, t) for t in range(3)] for i in range(3)
]
for group in groups:
    for k in group:
        a[k] -= penalty
    for pos, k in enumerate(group):
        for l in group[pos + 1 :]:
            add(k, l, 2 * penalty)

# 7. 전체 상태의 목적함수와 제약 검사
bits = (np.arange(2**n)[:, None] >> np.arange(n)) & 1
xmat = bits.reshape(-1, 3, 3)
row = xmat.sum(axis=2)
col = xmat.sum(axis=1)
valid = (row == 1).all(axis=1) & (col == 1).all(axis=1)
travel = np.zeros(2**n)
for i in range(3):
    travel += eta[0, i + 1] * bits[:, idx(i, 0)] + eta[i + 1, 0] * bits[:, idx(i, 2)]
for t in range(2):
    for i in range(3):
        for j in range(3):
            travel += eta[i + 1, j + 1] * bits[:, idx(i, t)] * bits[:, idx(j, t + 1)]
qubo = c + bits @ a + np.einsum("bi,ij,bj->b", bits, B, bits)
direct = travel + penalty * (
    ((row - 1) ** 2).sum(axis=1) + ((col - 1) ** 2).sum(axis=1)
)

# 8. QUBO를 Ising 계수로 변환
h = -a / 2 - (B.sum(axis=0) + B.sum(axis=1)) / 4
J = B / 4
constant = c + a.sum() / 2 + B.sum() / 4
z = 1 - 2 * bits
ising = constant + z @ h + np.einsum("bi,ij,bj->b", z, J, z)
assert valid.sum() == 6 and np.allclose(qubo, direct) and np.allclose(qubo, ising)
print(
    "QUBO 직접식/Ising 최대오차:",
    float(abs(qubo - direct).max()),
    float(abs(qubo - ising).max()),
)
# gamma(감마)는 비용 위상, beta(베타)는 믹서 회전량입니다.
# 비용을 penalty로 나누어 각도 탐색의 수치 범위를 조절합니다.


# 9. QAOA 회로 구성
def circuit(theta):
    qc = QuantumCircuit(n)
    qc.h(range(n))
    for layer in range(p):
        gamma, beta = theta[layer], theta[p + layer]
        for k in range(n):
            qc.rz(2 * gamma * h[k] / penalty, k)
        for k in range(n):
            for l in range(k + 1, n):
                if J[k, l] != 0:
                    qc.rzz(2 * gamma * J[k, l] / penalty, k, l)
        for k in range(n):
            qc.rx(2 * beta, k)
    return qc


# 각도 탐색 중 목적함수 값을 기록합니다.
history = []


def energy(theta):
    value = float(Statevector.from_instruction(circuit(theta)).probabilities() @ qubo)
    history.append(value)
    return value / penalty


# 10. 세 시작점에서 각도 최적화
runs = [
    minimize(
        energy,
        np.r_[np.full(p, g), np.full(p, b)],
        method="COBYLA",
        options={"maxiter": maxiter},
    )
    for g, b in [(0.2, 0.3), (0.7, 0.5), (1.2, 0.8)]
]
r = min(runs, key=lambda v: v.fun)

# 11. 측정과 최빈 상태 선택
sv = Statevector.from_instruction(circuit(r.x))
prob = sv.probabilities()
sv.seed(7)
counts = sv.sample_counts(shots)
observed = np.zeros(2**n, dtype=int)
for key, v in counts.items():
    observed[int(key, 2)] = v
chosen = int(observed.argmax())
best = float(travel[valid].min())
optimal = valid & np.isclose(travel, best, atol=1e-8, rtol=0)
assert np.isclose(prob.sum(), 1) and observed.sum() == shots

# 12. 목적함수·제약·관측 비율 출력
print("감마/베타:", r.x[:p], r.x[p:])
print("회로 평가 횟수:", sum(v.nfev for v in runs))
print("기대 에너지(상수 포함):", float(prob @ qubo))
print("선택 비트 k=0..8:", bits[chosen].tolist())
print("고객×순서 행렬:", xmat[chosen].tolist())
print("고객별 방문횟수:", row[chosen].tolist(), "순서별 배정수:", col[chosen].tolist())
print("제약 충족:", bool(valid[chosen]), "선택 QUBO:", float(qubo[chosen]))
print("원래 목적식:", float(travel[chosen]), "정확 최적 예측ETA(분):", best)
print(
    "실행 가능해 이론/관측 비율:",
    float(prob[valid].sum()),
    float(observed[valid].sum() / shots),
)
print(
    "최적해 이론/관측 비율:",
    float(prob[optimal].sum()),
    float(observed[optimal].sum() / shots),
)
print("선택 상태 이론/관측 비율:", float(prob[chosen]), float(observed[chosen] / shots))

# 13. 제약을 만족한 선택 상태의 배송 경로 복원
if valid[chosen]:
    route = [0] + [int(xmat[chosen, :, t].argmax()) + 1 for t in range(3)] + [0]
    print("측정 경로:", " → ".join(names[i] for i in route))
    print("최적값 차이(분):", float(travel[chosen] - best))
    print(
        "교육용 산식 ETA(분):",
        sum(reference[i, j] for i, j in zip(route[:-1], route[1:])),
    )
else:
    print("경로 복원: 제약 위반으로 배차 보류. 최적값 차이: 산정 제외")
print("정확 최적 비트:", bits[np.flatnonzero(optimal)].tolist())
print("차량 용량/시간창/교통 혼잡: 제약 및 입력 미설정")

# 14. 그래프와 측정 결과 저장
out = Path("day39_route")
out.mkdir(exist_ok=True)
fig, ax = plt.subplots(1, 2, figsize=(10, 3), facecolor="white")
ax[0].plot(losses, color="darkorange")
ax[0].set(xlabel="Epoch", ylabel="Scaled MSE")
ax[1].plot(history, color="darkorange")
ax[1].set(xlabel="Circuit evaluation", ylabel="QUBO expectation")
fig.tight_layout()
fig.savefig(out / "results.png", dpi=150)
plt.close(fig)
np.savez(
    out / "results.npz",
    eta=eta,
    bits=bits,
    probabilities=prob,
    observed=observed,
    selected=bits[chosen],
)
print("파일:", out / "results.png", out / "results.npz")
print("시간(초):", round(time.perf_counter() - started, 3))

버전: {'qiskit': '2.2.3', 'numpy': '2.4.6', 'scipy': '1.16.3', 'torch': '2.14.1+cpu', 'matplotlib': '3.10.0'}
학습 MSE(분²): 0.001593994558788836
ETA 행렬(분), 행=출발 열=도착 H,C1,C2,C3:
 [[ 0.     12.9077  6.8371 15.1619]
 [ 5.8777  0.      9.0091 11.7961]
 [ 5.3123 14.2839  0.     12.2256]
 [10.1819 13.6973  8.8051  0.    ]]
QUBO 직접식/Ising 최대오차: 0.0 0.0
감마/베타: [2.64308768] [0.42096105]
회로 평가 횟수: 121
기대 에너지(상수 포함): 291.89808809938546
선택 비트 k=0..8: [0, 1, 0, 0, 0, 1, 1, 0, 0]
고객×순서 행렬: [[0, 1, 0], [0, 0, 1], [1, 0, 0]]
고객별 방문횟수: [1, 1, 1] 순서별 배정수: [1, 1, 1]
제약 충족: True 선택 QUBO: 43.180460929870605
원래 목적식: 43.180460929870605 정확 최적 예측ETA(분): 38.63772249221802
실행 가능해 이론/관측 비율: 0.1840374219648761 0.1904296875
최적해 이론/관측 비율: 0.029252173472995684 0.030029296875
선택 상태 이론/관측 비율: 0.031319506575764794 0.0361328125
측정 경로: H → C3 → C1 → C2 → H
최적값 차이(분): 4.542738437652588
교육용 산식 ETA(분): 43.104835987091064
정확 최적 비트: [[0, 0, 1, 1, 0, 0, 0, 1, 0]]
차량 용량/시간창/교통 혼잡: 제약 및 입력 미설정
파일: day39_route/results.png day39_route

선택한 경로는 모든 고객을 한 번씩 방문하지만, 정확 최적 예측 ETA보다 약 4.54분 큽니다. 최빈 상태가 유효하다는 사실과 전체 측정 중 유효한 해의 비율은 각각 확인해야 합니다. 이 예에서는 유효한 해의 관측 비율이 약 19.04%입니다. 전체 측정에는 벌점이 큰 상태도 포함되므로 기대 에너지는 선택 경로 비용보다 크게 나옵니다.

선택 QUBO는 한 비트 상태의 벌점 포함 비용입니다. 제약을 충족하면 벌점이 0이어서 예측 ETA 합과 같습니다. 제약을 위반한 경우에는 출력된 목적식 값을 실제 경로 시간으로 해석하지 않습니다. 교육용 산식 ETA는 같은 경로에 라벨 생성 산식을 적용한 값이며 현장 실측 시간은 별도로 수집합니다.

정확 최적값 차이는 QAOA 선택 경로의 예측 ETA에서 유효 경로 중 최소 예측 ETA를 뺀 값입니다. 이론 확률은 상태벡터가 주는 값이고 관측 비율은 유한한 측정 횟수로 계산한 값입니다. 측정 시드와 횟수가 달라지면 최빈 상태가 달라질 수 있습니다.

`results.png`에는 학습 MSE와 QAOA 기대비용의 변화가 저장됩니다. `results.npz`에는 예측 ETA, 모든 비트 상태, 확률, 측정 횟수와 선택 상태가 저장됩니다. 서로 다른 시작점의 탐색이 이어지므로 기대비용 그래프의 경계에서 값이 바뀔 수 있습니다.